# Synthetic Runner

Interactive mirror of `mp_runners.synthetic_train_runner`.

## 0. Knobs

In [ ]:
RUN_FULL_RUN = True

KNOBS = {
    "device": "cuda",
    "out_dir": "outputs/runners/synthetic_train_runner_notebook",
    "run_name": "synthetic_notebook_check",
    "seed": 0,
    "ambient_dim": 2,
    "intrinsic_data_dim": 2,
    "n_train": 10,
    "n_test": 1280,
    "n_clusters": 3,
    "sigma_z": 0.10,
    "sigma_eta": 1.0e-3,
    "geometry_seed": 0,
    "hidden_dim": 128,
    "n_layers": 3,
    "time_emb_dim": 32,
    "prediction_space": "v",
    "loss_space": "v",
    "n_training_steps": 500000,
    "batch_cap": 10,
    "lr": 1.0e-3,
    "weight_decay": 0.0,
    "log_every": 10,
    "ckpt_every": 250,
    "eval_sample_bank_size": 64,
    "sampling_method": "heun",
    "sampling_steps": 32,
    "ratio_threshold": 0.3,
    "chunk_size": 256,
    "sliced_w2_projections": 64,
    "wandb": False,
    "wandb_mode": "offline",
    "wandb_project": "",
    "wandb_entity": "",
    "wandb_name": "",
    "wandb_group": "",
    "wandb_tags": "",
    "wandb_notes": "",
    "wandb_strict": False,
}

## 1. Imports

In [ ]:
import json
import sys
from pathlib import Path

import torch
from IPython.display import Image, display

NOTEBOOK_DIR = Path.cwd()
for candidate in (NOTEBOOK_DIR, *NOTEBOOK_DIR.parents):
    if (candidate / "src").is_dir() and (candidate / "prototyping").is_dir():
        REPO_ROOT = candidate
        break
else:
    raise RuntimeError("Could not find repo root")

for path in (REPO_ROOT, REPO_ROOT / "src"):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from gdfm import GDFM
from mp_config import config_to_dict, get_base_config
from mp_data import build_data_bundle
from mp_metrics.metrics_synthetic import (
    collect_controlled_synthetic_metric_record,
    memorization_metrics_in_spaces,
    gmm_intrinsic_metrics,
    project_to_subspace,
)
from mp_runners import synthetic_train_runner as runner
from mp_runners.common import count_parameters

In [ ]:
    argv = [
        "--device", str(knobs["device"]),
        "--out_dir", str(knobs["out_dir"]),
        "--run_name", str(knobs["run_name"]),
        "--seed", str(knobs["seed"]),
        "--ambient_dim", str(knobs["ambient_dim"]),
        "--intrinsic_data_dim", str(knobs["intrinsic_data_dim"]),
        "--n_train", str(knobs["n_train"]),
        "--n_test", str(knobs["n_test"]),
        "--n_clusters", str(knobs["n_clusters"]),
        "--sigma_z", str(knobs["sigma_z"]),
        "--sigma_eta", str(knobs["sigma_eta"]),
        "--geometry_seed", str(knobs["geometry_seed"]),
        "--hidden_dim", str(knobs["hidden_dim"]),
        "--n_layers", str(knobs["n_layers"]),
        "--time_emb_dim", str(knobs["time_emb_dim"]),
        "--prediction_space", str(knobs["prediction_space"]),
        "--loss_space", str(knobs["loss_space"]),
        "--n_training_steps", str(knobs["n_training_steps"]),
        "--batch_cap", str(knobs["batch_cap"]),
        "--lr", str(knobs["lr"]),
        "--weight_decay", str(knobs["weight_decay"]),
        "--log_every", str(knobs["log_every"]),
        "--ckpt_every", str(knobs["ckpt_every"]),
        "--eval_sample_bank_size", str(knobs["eval_sample_bank_size"]),
        "--sampling_method", str(knobs["sampling_method"]),
        "--sampling_steps", str(knobs["sampling_steps"]),
        "--ratio_threshold", str(knobs["ratio_threshold"]),
        "--chunk_size", str(knobs["chunk_size"]),
        "--sliced_w2_projections", str(knobs["sliced_w2_projections"]),
        "--wandb_mode", str(knobs["wandb_mode"]),
    ]
    if knobs["wandb"]:
        argv.append("--wandb")
    if knobs["wandb_project"]:
        argv.extend(["--wandb_project", str(knobs["wandb_project"])])
    if knobs["wandb_entity"]:
        argv.extend(["--wandb_entity", str(knobs["wandb_entity"])])
    if knobs["wandb_name"]:
        argv.extend(["--wandb_name", str(knobs["wandb_name"])])
    if knobs["wandb_group"]:
        argv.extend(["--wandb_group", str(knobs["wandb_group"])])
    if knobs["wandb_tags"]:
        argv.extend(["--wandb_tags", str(knobs["wandb_tags"])])
    if knobs["wandb_notes"]:
        argv.extend(["--wandb_notes", str(knobs["wandb_notes"])])
    if knobs["wandb_strict"]:
        argv.append("--wandb_strict")
    return argv

args = runner.parse_args(argv_from_knobs(KNOBS))
out_root = Path(args.out_dir).expanduser().resolve()
latest = out_root / "latest"
latest_checkpoints = out_root / "latest_checkpoints"
{"out_root": str(out_root), "latest": str(latest), "latest_checkpoints": str(latest_checkpoints)}

## 2. Config

In [ ]:
batching = {"batch_cap": args.batch_cap, "effective_batch_size": min(args.n_train, args.batch_cap), "drop_last": args.n_train > args.batch_cap}
effective_batch_size = min(args.n_train, args.batch_cap)
config = get_base_config()
config.study.name = runner.RUNNER_NAME
config.study.description = "Direct synthetic-data GDFM/MLP runner."
config.outputs.run_name = args.run_name
config.framework.family = "fm"
config.framework.prediction_space = args.prediction_space
config.framework.loss_space = args.loss_space
config.data.dataset = "gmm_subspace"
config.data.root = "synthetic"
config.data.download = False
config.data.data_source = "canonical"
config.data.n_train = args.n_train
config.data.n_test = args.n_test
config.data.ambient_dim = args.ambient_dim
config.data.intrinsic_data_dim = args.intrinsic_data_dim
config.data.data_dim = args.ambient_dim
config.data.intrinsic_dim = args.intrinsic_data_dim
config.data.n_clusters = args.n_clusters
config.data.sigma_z = args.sigma_z
config.data.sigma_eta = args.sigma_eta
config.data.geometry_seed = args.geometry_seed
config.data.flatten = False
config.network.arch = "mlp"
config.network.hidden_dim = args.hidden_dim
config.network.n_layers = args.n_layers
config.network.time_emb_dim = args.time_emb_dim
config.training.seed = args.seed
config.training.n_training_steps = args.n_training_steps
config.training.batch_cap = int(args.batch_cap)
config.training.batch_size = int(args.batch_cap)
config.training.lr = args.lr
config.training.weight_decay = args.weight_decay
config.training.ckpt_every = args.ckpt_every
config.training.log_every = args.log_every
config.training.optimizer = "adam"
config.evaluation.n_samples = args.eval_sample_bank_size
config.evaluation.ratio_threshold = args.ratio_threshold
config.evaluation.chunk_size = args.chunk_size
config.evaluation.sample_panel_count = min(8, args.eval_sample_bank_size)
config.sampling.method = args.sampling_method
config.sampling.n_steps = args.sampling_steps
config.sliced_w2_projections = args.sliced_w2_projections
config_payload = config_to_dict(config)
{
    "data": config_payload["data"],
    "network": config_payload["network"],
    "framework": config_payload["framework"],
    "training": config_payload["training"],
}

## 3. Data

In [ ]:
data_bundle = build_data_bundle(config)

train_tensor = data_bundle["train"].contiguous()
reference_tensor = data_bundle["reference"].contiguous()
train_labels = data_bundle["train_labels"]
reference_labels = data_bundle["reference_labels"]
basis = data_bundle["geometry"]["basis"].contiguous()
intrinsic_means = data_bundle["geometry"]["means"].contiguous()
ambient_means = intrinsic_means @ basis.T
{
    "train_shape": list(train_tensor.shape),
    "reference_shape": list(reference_tensor.shape),
    "basis_shape": list(basis.shape),
    
}

## 4. Projection And Baselines

In [ ]:
train_parts = project_to_subspace(train_tensor, basis)
reference_parts = project_to_subspace(reference_tensor, basis)
train_intrinsic = train_parts["intrinsic"]
reference_intrinsic = reference_parts["intrinsic"]

z_eval = torch.randn(args.eval_sample_bank_size, args.ambient_dim, generator=torch.Generator().manual_seed(args.seed))
sliced_generator = torch.Generator().manual_seed(args.seed + 1009)
sliced_directions = torch.randn(args.sliced_w2_projections, args.intrinsic_data_dim, generator=sliced_generator)
sliced_directions = sliced_directions / sliced_directions.norm(dim=1, keepdim=True).clamp(min=1.0e-12)

baseline_count = min(args.eval_sample_bank_size, reference_tensor.shape[0])
baseline_tensor = reference_tensor[:baseline_count]
baseline_intrinsic = reference_intrinsic[:baseline_count]
baseline_intrinsic_metrics = gmm_intrinsic_metrics(
    baseline_tensor, basis, intrinsic_means, args.sigma_z, args.sigma_eta, baseline_intrinsic, sliced_directions
)
baseline_mem = memorization_metrics_in_spaces(
    baseline_tensor, train_tensor, data_bundle["geometry"], train_intrinsic, chunk_size=args.chunk_size, config=config
)
{"baseline_mem": baseline_mem, "baseline_intrinsic": baseline_intrinsic_metrics}

## 5. Model And One Update

In [ ]:
device = args.device
if device == "auto":
    device = "cuda" if torch.cuda.is_available() else "cpu"
model = GDFM(config).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=args.lr, weight_decay=args.weight_decay)

batch = train_tensor[:effective_batch_size].to(device)
optimizer.zero_grad(set_to_none=True)
loss = model.compute_loss(batch)
loss.backward()
grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0e9)
optimizer.step()
{
    "device": device,
    "batch_shape": list(batch.shape),
    "loss": float(loss.detach().cpu().item()),
    "grad_norm": float(grad_norm.detach().cpu().item()),
    "parameters": count_parameters(model),
}

## 6. Metrics

In [ ]:
model.eval()
with torch.no_grad():
    samples = model.sample(
        z_eval.clone().to(device),
        n_steps=args.sampling_steps,
        method=args.sampling_method,
        device=device,
    ).cpu()
metric_record = collect_controlled_synthetic_metric_record(
    step=0,
    samples=samples,
    baseline_samples=baseline_tensor,
    train_tensor=train_tensor,
    train_intrinsic=train_intrinsic,
    geometry=data_bundle["geometry"],
    reference_intrinsic=baseline_intrinsic,
    sliced_directions=sliced_directions,
    chunk_size=args.chunk_size,
    config=config,
)
metric_record

## 7. Optional Full Run

In [ ]:
run_summary = None
if RUN_FULL_RUN:
    run_summary = runner.main(argv_from_knobs(KNOBS))
run_summary

## 8. Artifacts And Figures

In [ ]:
def load_latest_summary(root):
    latest_path = Path(root) / "latest"
    if latest_path.exists():
        summary_path = latest_path / "artifacts" / "run_summary.json"
        if summary_path.exists():
            return json.loads(summary_path.read_text()), latest_path
    return None, latest_path

summary, attempt_dir = (run_summary, Path(run_summary["run_dir"])) if run_summary else load_latest_summary(out_root)
if summary:
    print(json.dumps(summary.get("paths", {}), indent=2))
    for name, figure_path in summary.get("figures", {}).items():
        figure_path = Path(figure_path)
        if figure_path.suffix.lower() == ".png" and figure_path.exists():
            print(name, figure_path)
            display(Image(filename=str(figure_path)))
else:
    print(f"No completed synthetic run found at {attempt_dir}")